<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW5c_reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now, work only in the copy, and rename it
> `HW5c_LASTNAME.ipynb` (exact filename and due date in the assignment
> calendar on Brightspace).

# HW5c: Reflector Depth from the x²–t² Fit

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Due:** Sunday, December 13, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) to the Brightspace HW5c dropbox

**Extra credit:** This notebook is optional. It is graded out of 100 and counts as a bonus of up to 50 points in Problem Sets & Labs.

This notebook is new, and you are the first class to work it.

The mechanics of Colab (saving, running cells, errors, uploading) are on the **Working in Colab** page in the Resources module on Brightspace.

This assignment builds on the Wednesday, December 2 class. It uses Burger section 4.1.

## What you will do

1. Plot a real record as wiggle traces, with its reflection times.
2. Fit the survey and find the depth to the reflector.
3. Compare the normal move-out for a shallow and a deeper reflector.

## What you will hand in

This same notebook with every cell run and the answer cells filled in. Each question has one answer cell that opens with its label. Double-click the cell and type under the label.

## Feedback

After each section there is a feedback cell, and one more at the end. Tell me what worked, what did not, and where you got stuck. Write it as you go, while it is fresh. The feedback cells carry no points.

## Learning objectives

This notebook builds these course objectives:

10. Compute the reflection time from one flat interface at a given distance. Sketch the reflection curve beside the direct wave on a time-distance plot. Explain why the reflection is never the first arrival.
11. Compute the normal move-out at a given distance as the reflection time there minus the zero-offset time. Explain why it gets smaller as the reflector gets deeper.
12. Plot t² against x² for the reflection times and fit a straight line. Find the layer velocity from the slope and the zero-offset time from the intercept. Compute the depth to the reflector.

## Data source and citation

> Pratt, T.L., 2026, Seismic reflection profiles across the Puente Hills fault zone, Los Angeles, California: U.S. Geological Survey data release, https://doi.org/10.5066/P1XIR92T. Public domain (CC0).

The U.S. Geological Survey shot this line along Carfax Avenue in Los Angeles in 2000. The source is a sledgehammer striking the ground at 0 m. The geophones were set out 1 m apart. Distances are measured along the line. The field notes put the hammer 2 m to the side of the line. Both files below hold one shot, field file 4031.

`reflection_puente_hills_ffid4031.csv` holds the reflection times. At each geophone, the arrival time of the wave reflected from a layer boundary below the street was picked from the record. The course made the picks from the published record. Geophones with a dead trace or no clear arrival are left out, so the file keeps 43 geophones from 6 m to 59 m, mostly 1 m apart with a few gaps of up to 4 m.

| Column | Meaning |
|---|---|
| `offset_m` | Distance from the source to the geophone (m) |
| `reflection_time_ms` | Reflection time (ms) |
| `travel_time_s` | The same time in seconds |

`reflection_puente_hills_ffid4031_traces.csv` holds the record itself: the ground motion at each geophone for the first 200 ms after the hammer blow, one value every 0.5 ms. The course kept the frequencies from 60 to 300 Hz. The values keep the recorder's units, so the size of one trace compares with the size of another. The course dropped traces with less than a tenth of the typical trace's energy. The file keeps 46 geophones from 6 m to 59 m.

| Column | Meaning |
|---|---|
| `time_ms` | Time after the hammer blow (ms) |
| `6` to `59` | One column per geophone, named by its distance from the source (m): the ground motion in recorder units |

## Loading the data

The code below loads `reflection_puente_hills_ffid4031.csv` and `reflection_puente_hills_ffid4031_traces.csv` automatically from a stable public web address, so in most cases you do not need to download anything: run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download both files from the Brightspace HW5c page.
2. In Google Colab, click the **Files** icon in the left sidebar.
3. Drag both files into the file panel.
4. In each loading cell below, comment out the `pd.read_csv` line that reads a web address and use the commented line that reads the file by name.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

## Setup

These imports load NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

The cell also defines `run_check`. A run-check cell prints one line that reads PASS or FAIL. A run check carries no question and no points. It confirms that the cells above ran as written. If one reads FAIL, the problem is upstream of it.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe
COLORS = px.colors.qualitative.Safe


def run_check(name, passed, detail):
    """Print one run-check line: PASS, or FAIL with the value that missed."""
    print(f'{name} {"PASS" if passed else "FAIL"}: {detail}')


**Load.** Reads `reflection_puente_hills_ffid4031.csv` into `df` and defines `x`, the distance from the source, and `t`, the reflection time in seconds. The printout gives the number of geophones, the range of distances and the spacing.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/reflection_puente_hills_ffid4031.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download
# reflection_puente_hills_ffid4031.csv from the Brightspace HW5c page, drag it into
# the Colab file panel, then comment out the df = pd.read_csv(DATA_URL) line above
# and use:
# df = pd.read_csv("reflection_puente_hills_ffid4031.csv")

x = df['offset_m'].to_numpy()          # distance from the source (m)
t = df['travel_time_s'].to_numpy()     # reflection time (s)
print(f'{len(df)} geophones, {x.min():.0f} m to {x.max():.0f} m, '
      f'{np.diff(x).min():.0f} to {np.diff(x).max():.0f} m apart')
df.head(10)


**Load the record.** Reads `reflection_puente_hills_ffid4031_traces.csv` into `traces` and defines `time_ms`, the time after the hammer blow. The printout gives the number of traces, the time range and the time between values.

In [ ]:
# The record itself: one column of ground motion for each geophone.
TRACES_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/reflection_puente_hills_ffid4031_traces.csv"
traces = pd.read_csv(TRACES_URL)

# Fallback (no internet, or URL not live yet): download
# reflection_puente_hills_ffid4031_traces.csv from the Brightspace HW5c page, drag it
# into the Colab file panel, then comment out the traces = pd.read_csv(TRACES_URL)
# line above and use:
# traces = pd.read_csv("reflection_puente_hills_ffid4031_traces.csv")

time_ms = traces['time_ms'].to_numpy()   # time after the hammer blow (ms)
print(f'{traces.shape[1] - 1} traces, {time_ms.min():.0f} ms to {time_ms.max():.0f} ms, '
      f'one value every {time_ms[1] - time_ms[0]:.1f} ms')


## Reflection times and the x²–t² method

This cell plots the record as wiggle traces. Each trace is the ground motion at one geophone, drawn around that geophone's distance from the source, with time increasing up the page. The positive lobes are filled black. The circles and the thin line are the reflection times from the first file. The dashed line is the air wave.

In [ ]:
import matplotlib.pyplot as plt

# One gain for the whole record: every trace is multiplied by the same number.
gain = time_ms / 100                           # later times are multiplied by more
SHOWN = [c for c in traces.columns[1:] if 20 <= float(c) <= 48]   # clean reflection: 20 m to 48 m
motions = traces[SHOWN].to_numpy() * gain[:, None]
scale = np.percentile(np.abs(motions), 90)     # one scale factor for all plotted traces
SWING = 1.0                                    # a swing of one scale unit is drawn 1 m wide

fig, ax = plt.subplots(figsize=(10, 7))
for column in SHOWN:
    offset = float(column)                     # this geophone's distance from the source (m)
    motion = np.clip(traces[column].to_numpy() * gain / scale, -1, 1)   # clip large swings
    wiggle = offset + SWING * motion           # draw the trace around its own distance
    ax.plot(wiggle, time_ms, color='black', linewidth=0.5)
    ax.fill_betweenx(time_ms, offset, wiggle, where=motion > 0, color='black')  # fill the positive lobes

# The reflection picks from the first file, with a thin line joining them.
shown_picks = df[(df['offset_m'] >= 20) & (df['offset_m'] <= 48)]
ax.plot(shown_picks['offset_m'], shown_picks['reflection_time_ms'], color='#E69F00', linewidth=1)
ax.plot(shown_picks['offset_m'], shown_picks['reflection_time_ms'], linestyle='none', marker='o',
        markersize=5, markerfacecolor='none', markeredgecolor='#E69F00',
        markeredgewidth=1.5, label='reflection picks')

# The air wave: a sound wave through the air at 343 m/s, t = x / 343.
x_air = np.array([18, 50])
ax.plot(x_air, x_air / 343 * 1000, color='#0072B2', linewidth=1.5,
        linestyle='--', label='air wave, 343 m/s')

ax.set_xlim(18, 50)
ax.set_ylim(0, 200)   # time increases up the page, as on the time-distance plots
ax.set_xlabel('Distance from the source (m)')
ax.set_ylabel('Time after the hammer blow (ms)')
ax.set_title('Shot 4031 as wiggle traces, with the reflection picks')
ax.legend(loc='lower right')
plt.show()


**Figure description:** One wavy trace per geophone, distance from the source (18 m to 50 m, x-axis) against time after the hammer blow (0 to 200 ms, y-axis, increasing upward). Traces closer than 20 m are covered by strong, slow ground roll, and traces beyond 48 m are faint, so the plot shows 20 to 48 m. Filled black lobes are ground motion in one direction. All traces share one gain: each value is multiplied by the time in ms divided by 100, divided by one scale factor, and clipped at one swing. Open orange circles joined by a thin orange line mark the reflection picks in that range. The line starts near 85 ms at 21 m and curves up to about 152 ms at 48 m. Strong motion begins along that curve. A dashed blue straight line, the air wave at 343 m/s, runs from about 52 ms at 18 m to about 146 ms at 50 m. The curved reflection and the straight air wave are separate events. Non-visual path: the table printed under the next plot lists every pick.

This cell plots the reflection time at each geophone against its distance from the source, then prints the same numbers as a table.

In [ ]:
fig = px.scatter(df, x='offset_m', y='reflection_time_ms',
                 labels={'offset_m': 'Distance from the source (m)',
                         'reflection_time_ms': 'Reflection time (ms)'},
                 title='Reflection times against distance')
fig.update_traces(marker=dict(size=9, symbol='circle'))
fig.update_yaxes(rangemode='tozero')
fig.show()

# Non-visual path: every geophone as a table.
print(df[['offset_m', 'reflection_time_ms']].to_string(index=False))


**Figure description:** A scatter plot of reflection time (ms, y-axis) against distance from the source (m, x-axis), one circle per geophone. The times run from about 63 ms at 6 m to about 181 ms at 59 m. Near the source the points rise slowly. Farther out they climb more steeply, so together they trace a curve. Non-visual path: the table printed under the plot lists every geophone.

**The reflection travel time.** A reflected wave travels down to the reflector and back up, at velocity $V_1$ the whole way. For a flat reflector at depth $h_1$ and a geophone at distance $x$ from the source, Burger eq 4.1 (section 4.1.2) gives its travel time:

$$t = \frac{\sqrt{x^2 + 4h_1^2}}{V_1}$$

Square both sides. The result has the form of a straight line, $y = mx + b$, with $x^2$ in place of $x$ and $t^2$ in place of $y$ (eq 4.6):

$$t^2 = \frac{1}{V_1^2}\,x^2 + \frac{4h_1^2}{V_1^2}$$

The slope is $1/V_1^2$. The intercept is $t_0^2$, where $t_0$ is the reflection time at $x = 0$:

$$t_0 = \frac{2h_1}{V_1} \qquad h_1 = \frac{t_0 V_1}{2} \quad \text{(eq 4.7)}$$

This is the **x²–t² method** (section 4.1.5). Fit a straight line to $t^2$ against $x^2$. Take $V_1$ from the slope and $t_0$ from the intercept. Then eq 4.7 gives $h_1$.

**Units.** Burger works this method in meters and milliseconds. The slope then comes out in ms² per m², and $1/\sqrt{\text{slope}}$ is a velocity in meters per millisecond. Use the velocity in m/ms in eq 4.7 when $t_0$ is in ms. Multiply it by 1000 to report it in m/s.

**Fit the survey.** This cell fits a straight line to $t^2$ against $x^2$, in milliseconds. It takes $V_1$ from the slope and $t_0$ from the intercept, then applies eq 4.7.

In [ ]:
t_ms = df['reflection_time_ms'].to_numpy()   # reflection time (ms), the unit Burger uses

# One straight line through t squared against x squared (Burger eq 4.6).
slope, intercept = np.polyfit(x**2, t_ms**2, 1)

v1_m_per_ms = 1 / np.sqrt(slope)      # m per ms
v1 = v1_m_per_ms * 1000               # m/s
t0 = np.sqrt(intercept)               # zero-offset time (ms)
h1 = t0 * v1_m_per_ms / 2             # Burger eq 4.7 (m)

print(f'slope = {slope:.4f} ms^2 per m^2   intercept = {intercept:.0f} ms^2')
print(f'V1 = {v1:.0f} m/s')
print(f't0 = {t0:.1f} ms')
print(f'depth to the reflector h1 = {h1:.1f} m')


$V_1$ comes out near 343 m/s. This is close to the speed of sound in air. The USGS processing of this record uses 300 to 350 m/s for the shallow ground.

This cell plots $t^2$ against $x^2$ with the fitted line.

In [ ]:
x2_line = np.linspace(0.0, (x**2).max(), 200)
fig = go.Figure()
fig.add_scatter(x=x**2, y=t_ms**2, mode='markers', name='reflection times, squared',
                marker=dict(size=9, symbol='circle', color=COLORS[0]))
fig.add_scatter(x=x2_line, y=slope * x2_line + intercept, mode='lines',
                name=f'fitted line, V1 = {v1:.0f} m/s',
                line=dict(color=COLORS[3], width=2, dash='solid'))
fig.update_xaxes(title_text='Distance squared (m²)')
fig.update_yaxes(title_text='Reflection time squared (ms²)', rangemode='tozero')
fig.update_layout(title='t² against x² with the fitted line')
fig.show()


**Figure description:** Forty-three circles, reflection time squared (ms², y-axis) against distance squared (m², x-axis), with the fitted straight line drawn from zero. The line meets the time axis near 3440 ms² and rises to about 33,000 ms² at 3481 m². The circles scatter evenly about the line. Non-visual path: the fit cell prints the slope, the intercept, the velocity, the zero-offset time and the depth.

**Run check 1.** $V_1$ should fall between 330 and 356 m/s, $t_0$ between 55 and 62 ms and the depth between 9.5 and 10.6 m. If this reads FAIL, choose Runtime > Restart and run all.

In [ ]:
run_check('Run check 1',
          330 <= v1 <= 356 and 55 <= t0 <= 62 and 9.5 <= h1 <= 10.6,
          f'V1 {v1:.0f} m/s (330 to 356); t0 {t0:.1f} ms (55 to 62); '
          f'h1 {h1:.1f} m (9.5 to 10.6)')


**Question 1.** Report $V_1$ and $t_0$. Report the depth to the reflector. The slope is in ms² per m². What unit does $1/\sqrt{\text{slope}}$ carry? What depth would eq 4.7 give if you used $V_1$ in m/s with $t_0$ in ms? (Burger section 4.1.5.)

**Q1 answer:**



**Question 2.** $t_0$ is the reflection time at zero distance from the source. Describe the path the wave travels in that time. The reflection times change very little across the first few geophones. Explain why from eq 4.1. Compare $x^2$ with $4h_1^2$ at the 6 m geophone.

**Q2 answer:**



**Feedback on this section.** What worked? What did not? Where did you get stuck?

**Feedback 1:**



## Normal move-out

At every geophone past the source, the reflection arrives later than $t_0$. Burger calls the extra time the **normal move-out** (section 4.1.4). It is eq 4.1 minus the zero-offset time (eq 4.5):

$$T_{NMO} = \frac{\sqrt{x^2 + 4h_1^2}}{V_1} - \frac{2h_1}{V_1}$$

This cell uses your $V_1$ to find the normal move-out at the farthest geophone for two reflectors: one at the depth you found and one at twice that depth.

In [ ]:
x_far = x.max()   # the farthest geophone (m)

for depth in [h1, 2 * h1]:
    t_zero = 2 * depth / v1_m_per_ms                            # zero-offset time (ms)
    t_far = np.sqrt(x_far**2 + 4 * depth**2) / v1_m_per_ms      # Burger eq 4.1 (ms)
    nmo = t_far - t_zero                                        # Burger eq 4.5 (ms)
    print(f'reflector at {depth:.1f} m: t0 = {t_zero:.1f} ms, '
          f'time at {x_far:.0f} m = {t_far:.1f} ms, '
          f'normal move-out = {nmo:.1f} ms')


**Question 3.** Report the normal move-out at 59 m for the depth you found and for twice that depth. Which is smaller? Use eq 4.1 and eq 4.5 to explain why. (Burger section 4.1.4.)

**Q3 answer:**



**Feedback on this section.** What worked? What did not? Where did you get stuck?

**Feedback 2:**



**Feedback on the whole notebook.** What would you change before another class works this notebook?

**Feedback 3:**



## How to submit

1. Run all cells from the top. (`Runtime > Run all` in Colab.) Every run-check line reads PASS.
2. Make sure every answer cell is filled in.
3. `File > Download > Download .ipynb`.
4. Rename the file to `HW5c_LASTNAME.ipynb` (for example `HW5c_Smith.ipynb`).
5. Upload it to the Brightspace HW5c dropbox by **Sunday, December 13, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.